# 04A — Final Test Candidate Generation — Zero-Shot — FINAL V1

Notebook ini **hanya membangkitkan kandidat final** untuk skenario Zero-Shot.

- Backbone: `meta-llama/Llama-3.1-8B-Instruct`
- Test: **53 input**
- Kandidat per input: **5**
- Seed: `42, 43, 44, 45, 46`
- `temperature=0.70`, `top_p=0.95`, `top_k=0`, `num_beams=1`
- `max_new_tokens=128`
- BF16, tanpa kuantisasi, tanpa adapter
- **Reference Indonesia test tidak dibaca sama sekali** (`usecols` hanya metadata + `duri_text`)
- Notebook ini **belum menghitung BLEU/chrF++, S3E, atau MBR**. Itu dilakukan pada tahap berikutnya.

Output utama: `Hasil/final_generation/zero_shot/run_<id>/candidates_long.csv`.


## 1. Install dependency

In [1]:
# ============================================================
# INSTALL — jalankan sekali pada instance Vast.ai baru
# ============================================================
# PyTorch/CUDA bawaan image TIDAK diinstal ulang.
%pip -q install -U --upgrade-strategy only-if-needed \
    "transformers==4.48.1" \
    "accelerate==1.2.1" \
    "huggingface_hub==0.27.1" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "tqdm>=4.66,<5"

print('Dependency selesai.')
print('Jika package berubah dan kernel meminta restart, restart SATU KALI lalu mulai lagi dari Environment.')


Note: you may need to restart the kernel to use updated packages.
Dependency selesai.
Jika package berubah dan kernel meminta restart, restart SATU KALI lalu mulai lagi dari Environment.


## 2. Environment + test-input audit + GPU preflight

In [2]:
from __future__ import annotations

import os, re, json, time, random, hashlib, gc, subprocess
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# ============================================================
# PROJECT LOCK
# ============================================================
PROJECT_ROOT = Path(
    os.getenv(
        'SKRIPSI_PROJECT_ROOT',
        '/workspace/10122233_CikitaMeydilvira_Skripsi',
    )
).expanduser().resolve()

DATASET_DIR = PROJECT_ROOT / 'Dataset'
TRAIN_PATH = DATASET_DIR / 'train.csv'
TEST_PATH = DATASET_DIR / 'test.csv'
CONFIG_DIR = PROJECT_ROOT / 'Config'
HASIL_DIR = PROJECT_ROOT / 'Hasil'

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
DEVICE_INDEX = 0
DTYPE = torch.bfloat16

EXPECTED_TRAIN = 417
EXPECTED_TRAIN_CONV = 44
EXPECTED_TEST = 53
EXPECTED_TEST_CONV = 3

# ============================================================
# FINAL MULTI-INSTANCE GENERATION LOCK
# ============================================================
CANDIDATE_SEEDS = [42, 43, 44, 45, 46]
TEMPERATURE = 0.70
TOP_P = 0.95
TOP_K = 0               # disable top-k; sampling dikontrol temperature + top_p
NUM_BEAMS = 1
MAX_NEW_TOKENS = 128
REPETITION_PENALTY = 1.0

os.environ.setdefault('HF_HOME', '/workspace/.hf_home')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

HF_TOKEN = get_token() or os.getenv('HF_TOKEN', '').strip() or None

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}')

if not TEST_PATH.exists():
    raise FileNotFoundError(f'test.csv tidak ditemukan: {TEST_PATH}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak tersedia.')

torch.cuda.set_device(DEVICE_INDEX)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError('GPU tidak mendukung BF16.')

if not HF_TOKEN:
    raise RuntimeError('Hugging Face belum login. Jalankan `hf auth login` di terminal.')

# ------------------------------------------------------------
# GPU preflight: notebook final inference harus dimulai dari GPU bersih.
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
free_gib = free_b / (1024 ** 3)
total_gib = total_b / (1024 ** 3)
print(f'GPU preflight: {free_gib:.2f} GiB free / {total_gib:.2f} GiB total')

if free_gib < 20.0:
    raise RuntimeError(
        f'VRAM bebas hanya {free_gib:.2f} GiB. '
        'Shutdown kernel notebook lain sebelum menjalankan final generation.'
    )

# ------------------------------------------------------------
# Test loader: target/reference SENGAJA tidak dibaca.
# ------------------------------------------------------------
TEST_INPUT_COLUMNS = [
    'row_id',
    'conversation_no',
    'turn_order',
    'duri_text',
]

test_df = pd.read_csv(
    TEST_PATH,
    usecols=TEST_INPUT_COLUMNS,
    dtype=str,
    keep_default_na=False,
    encoding='utf-8-sig',
)

test_df.columns = [str(c).strip() for c in test_df.columns]
for c in TEST_INPUT_COLUMNS:
    test_df[c] = test_df[c].astype(str).str.strip()

if len(test_df) != EXPECTED_TEST:
    raise ValueError(f'test.csv expected {EXPECTED_TEST}, found {len(test_df)}')

if test_df['row_id'].eq('').any() or test_df['row_id'].duplicated().any():
    raise ValueError('test.csv: row_id kosong/duplikat.')

if test_df['duri_text'].eq('').any():
    raise ValueError('test.csv: terdapat source Duri kosong.')

if test_df['conversation_no'].nunique() != EXPECTED_TEST_CONV:
    raise ValueError(
        f'Test conversation expected {EXPECTED_TEST_CONV}, '
        f'found {test_df["conversation_no"].nunique()}'
    )


def norm_text(x: str) -> str:
    return re.sub(r'\s+', ' ', str(x).strip().casefold())


def test_input_hash() -> str:
    h = hashlib.sha256()
    ordered = test_df.sort_values('row_id', kind='stable')
    for r in ordered.itertuples(index=False):
        payload = {
            'row_id': r.row_id,
            'conversation_no': r.conversation_no,
            'turn_order': r.turn_order,
            'duri_text': r.duri_text,
        }
        h.update(
            (json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(',', ':')) + '\n')
            .encode('utf-8')
        )
    return h.hexdigest()

TEST_INPUT_HASH = test_input_hash()


def terminator_ids(tokenizer) -> List[int]:
    ids = []
    if tokenizer.eos_token_id is not None:
        if isinstance(tokenizer.eos_token_id, (list, tuple)):
            ids.extend(int(x) for x in tokenizer.eos_token_id)
        else:
            ids.append(int(tokenizer.eos_token_id))

    for tok in ['<|eot_id|>', '<|end_of_text|>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if (
            isinstance(tid, int)
            and tid >= 0
            and tid != tokenizer.unk_token_id
            and tid not in ids
        ):
            ids.append(tid)
    return ids


def clean_translation(text: str) -> str:
    x = re.sub(r'\s+', ' ', str(text or '').strip())
    prefixes = [
        'Bahasa Indonesia:',
        'Indonesia:',
        'Terjemahan:',
        'Hasil terjemahan:',
        'Jawaban:',
    ]
    changed = True
    while changed:
        changed = False
        for p in prefixes:
            if x.casefold().startswith(p.casefold()):
                x = x[len(p):].strip()
                changed = True
                break
    return x


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

print('PROJECT_ROOT   :', PROJECT_ROOT)
print('TEST           :', len(test_df), 'rows /', test_df['conversation_no'].nunique(), 'conversation')
print('REFERENCE      : TIDAK DIBACA')
print('TEST INPUT HASH:', TEST_INPUT_HASH)
print('GPU            :', torch.cuda.get_device_name(DEVICE_INDEX))
print('MODEL          :', MODEL_ID)
print('SEEDS          :', CANDIDATE_SEEDS)
print('SAMPLING       : T=0.70, top_p=0.95, top_k=0, beams=1')


GPU preflight: 22.75 GiB free / 23.52 GiB total
PROJECT_ROOT   : /workspace/10122233_CikitaMeydilvira_Skripsi
TEST           : 53 rows / 3 conversation
REFERENCE      : TIDAK DIBACA
TEST INPUT HASH: 4d3e338a3effb23b17c0dca790808cd031072969484af3ffcb4f1bb9698242d2
GPU            : NVIDIA GeForce RTX 4090
MODEL          : meta-llama/Llama-3.1-8B-Instruct
SEEDS          : [42, 43, 44, 45, 46]
SAMPLING       : T=0.70, top_p=0.95, top_k=0, beams=1


## 3. Load Llama

In [3]:
# ============================================================
# LOAD LLAMA 3.1 8B INSTRUCT — BF16, no quantization
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map={'': DEVICE_INDEX},
)

model.eval()
model.config.use_cache = True
EOS_IDS = terminator_ids(tokenizer)

free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
print('Llama loaded.')
print(f'Free VRAM: {free_b/(1024**3):.2f} GiB / {total_b/(1024**3):.2f} GiB')
print('Terminator IDs:', EOS_IDS)


model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Llama loaded.
Free VRAM: 8.17 GiB / 23.52 GiB
Terminator IDs: [128009, 128001]


## 4. Zero-Shot prompt + generation helper

In [4]:
# ============================================================
# ZERO-SHOT PROMPT
# Tidak ada example dan tidak ada adapter.
# ============================================================
ZERO_SHOT_INSTRUCTION = (
    'Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia. '
    'Keluarkan hanya terjemahan bahasa Indonesia tanpa penjelasan tambahan.'
)


def build_zero_shot_prompt(source: str) -> str:
    return '\n'.join([
        ZERO_SHOT_INSTRUCTION,
        '',
        'Teks yang harus diterjemahkan:',
        f'Duri: {source}',
        'Indonesia:',
    ]).strip()


def encode_prompt(prompt: str) -> Dict[str, torch.Tensor]:
    enc = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    )
    return {k: v.to(f'cuda:{DEVICE_INDEX}') for k, v in enc.items()}


def generate_one_candidate(
    enc: Dict[str, torch.Tensor],
    seed: int,
) -> Tuple[str, int, float]:
    seed_everything(seed)
    prompt_tokens = int(enc['input_ids'].shape[-1])

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            num_beams=NUM_BEAMS,
            repetition_penalty=REPETITION_PENALTY,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=EOS_IDS if EOS_IDS else None,
        )

    torch.cuda.synchronize()
    latency = time.perf_counter() - t0

    completion_tokens = int(out.shape[-1] - prompt_tokens)
    raw = tokenizer.decode(
        out[0, prompt_tokens:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return clean_translation(raw), completion_tokens, latency


## 5. Generate 53 × 5 candidates + save

In [5]:
# ============================================================
# FINAL ZERO-SHOT — 53 test × 5 candidates
# Reference test TIDAK digunakan atau disimpan di output generation.
# ============================================================
run_payload = {
    'scenario': 'zero_shot',
    'model_id': MODEL_ID,
    'test_input_hash': TEST_INPUT_HASH,
    'reference_read': False,
    'candidate_seeds': CANDIDATE_SEEDS,
    'generation': {
        'do_sample': True,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'top_k': TOP_K,
        'num_beams': NUM_BEAMS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'repetition_penalty': REPETITION_PENALTY,
    },
    'prompt_version': 'zero_shot_duri_indonesia_final_v1',
}

RUN_ID = hashlib.sha256(
    json.dumps(run_payload, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()[:16]

RUN_DIR = HASIL_DIR / 'final_generation' / 'zero_shot' / f'run_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = RUN_DIR / 'candidates_long.csv'
WIDE_PATH = RUN_DIR / 'candidates_wide.csv'
MANIFEST_PATH = RUN_DIR / 'run_manifest.json'
SUMMARY_PATH = RUN_DIR / 'generation_summary.json'

MANIFEST_PATH.write_text(
    json.dumps({**run_payload, 'run_id': RUN_ID, 'created_at_utc': utc_now_iso()}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

if CANDIDATES_PATH.exists():
    records = pd.read_csv(
        CANDIDATES_PATH,
        dtype={'row_id': str, 'seed': int},
        keep_default_na=False,
        encoding='utf-8-sig',
    ).to_dict('records')
else:
    records = []

done = {(str(r['row_id']), int(r['seed'])) for r in records if str(r.get('output', '')).strip()}

for row in tqdm(test_df.itertuples(index=False), total=len(test_df), desc='Zero-Shot test'):
    source = str(row.duri_text)
    prompt = build_zero_shot_prompt(source)
    enc = encode_prompt(prompt)
    prompt_tokens = int(enc['input_ids'].shape[-1])

    for candidate_id, seed in enumerate(CANDIDATE_SEEDS, start=1):
        key = (str(row.row_id), int(seed))
        if key in done:
            continue

        error = ''
        output = ''
        completion_tokens = 0
        latency = np.nan

        try:
            output, completion_tokens, latency = generate_one_candidate(enc, seed)
            if not output:
                raise RuntimeError('Output kosong.')
        except Exception as e:
            error = f'{type(e).__name__}: {e}'

        records.append({
            'scenario': 'zero_shot',
            'row_id': str(row.row_id),
            'conversation_no': str(row.conversation_no),
            'turn_order': str(row.turn_order),
            'source': source,
            'candidate_id': candidate_id,
            'seed': seed,
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
            'top_k': TOP_K,
            'max_new_tokens': MAX_NEW_TOKENS,
            'prompt_tokens': prompt_tokens,
            'completion_tokens': completion_tokens,
            'latency_seconds': latency,
            'output': output,
            'error': error,
        })

        # checkpoint setelah setiap kandidat
        ckpt = pd.DataFrame(records).drop_duplicates(
            subset=['row_id', 'seed'], keep='last'
        ).sort_values(['row_id', 'candidate_id'], kind='stable')
        ckpt.to_csv(CANDIDATES_PATH, index=False, encoding='utf-8-sig')

        if output:
            done.add(key)

    del enc
    gc.collect()

cand = pd.read_csv(CANDIDATES_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
cand['candidate_id'] = pd.to_numeric(cand['candidate_id'], errors='raise').astype(int)

success = cand['output'].astype(str).str.strip().ne('')
expected_total = EXPECTED_TEST * len(CANDIDATE_SEEDS)

if len(cand) != expected_total or int(success.sum()) != expected_total:
    bad = cand.loc[~success, ['row_id', 'candidate_id', 'seed', 'error']]
    if not bad.empty:
        display(bad)
    raise RuntimeError(
        f'Final Zero-Shot belum lengkap: rows={len(cand)}, success={int(success.sum())}, expected={expected_total}. '
        f'Run ulang notebook; progress akan di-resume dari {CANDIDATES_PATH}.'
    )

wide = cand.pivot(index='row_id', columns='candidate_id', values='output').reset_index()
wide = wide.rename(columns={i: f'candidate_{i}' for i in range(1, 6)})
wide = test_df[['row_id', 'conversation_no', 'turn_order', 'duri_text']].merge(wide, on='row_id', how='left')
wide.to_csv(WIDE_PATH, index=False, encoding='utf-8-sig')

summary = {
    'scenario': 'zero_shot',
    'run_id': RUN_ID,
    'test_rows': EXPECTED_TEST,
    'candidates_per_row': len(CANDIDATE_SEEDS),
    'candidate_rows': int(len(cand)),
    'complete': True,
    'reference_read': False,
    'test_input_hash': TEST_INPUT_HASH,
    'candidates_path': str(CANDIDATES_PATH),
    'wide_path': str(WIDE_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nZERO-SHOT FINAL GENERATION: COMPLETE 53 × 5')


Zero-Shot test:   0%|          | 0/53 [00:00<?, ?it/s]

{
  "scenario": "zero_shot",
  "run_id": "1ef71f2034f92d52",
  "test_rows": 53,
  "candidates_per_row": 5,
  "candidate_rows": 265,
  "complete": true,
  "reference_read": false,
  "test_input_hash": "4d3e338a3effb23b17c0dca790808cd031072969484af3ffcb4f1bb9698242d2",
  "candidates_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/zero_shot/run_1ef71f2034f92d52/candidates_long.csv",
  "wide_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/zero_shot/run_1ef71f2034f92d52/candidates_wide.csv"
}

ZERO-SHOT FINAL GENERATION: COMPLETE 53 × 5
